# 方案二：Double-Propagation 规则基线复现

本方案不需要训练神经网络：先用训练标签归纳"观点词→极性 / 方面词→类别"知识库，再借助 **spaCy 中文依存句法** 在观点词与方面词之间双向传播（R1~R4 规则），最后用规则流水线组装四元组。

| 指标 | dev（484 条，可本地复算） | 测试集（官方排行榜） |
| --- | --- | --- |
| 四元组精确匹配 F1 | **0.5535** | **0.4925** |
| (A, O) 对 F1 | 0.5850 | — |
| 方面词 F1 / 观点词 F1 | 0.7947 / 0.7027 | — |

> 运行时间：CPU 约 1 分钟（规则基线不需要 GPU）。脚本产物路径全部写入 `src/config.py` 规定的位置（项目根目录 `data/baseline/`、`outputs/logs/`）。

## Step 0. 环境与依赖

首次运行请取消注释安装依赖（**国内机器先配清华镜像可加速 5~10 倍**）：

```bash
# 一次性配置清华镜像
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
pip config set global.trusted-host pypi.tuna.tsinghua.edu.cn

# 主依赖（pandas / numpy / jieba / matplotlib / spacy / PyYAML）
pip install -r requirements.txt

# spaCy 中文句法模型（3.8.7 的 spacy 对应 3.8.0 的 zh_core_web_sm）
# 不走 python -m spacy download（慢 + 不稳定），直接装 wheel：
pip install https://github.com/explosion/spacy-models/releases/download/zh_core_web_sm-3.8.0/zh_core_web_sm-3.8.0-py3-none-any.whl
```

EDA 画图使用 matplotlib；Windows 下中文显示依赖 SimHei / Microsoft YaHei 字体。

In [ ]:
import importlib.util
if importlib.util.find_spec("spacy") is None:
    print("spaCy 未安装，请先运行 Step 0 的 pip install 命令")
else:
    import spacy
    try:
        nlp = spacy.load("zh_core_web_sm")
        print("spaCy 中文模型 zh_core_web_sm: 可用")
        print("  示例依存:")
        for tok in nlp("物流很快，但是遮瑕一般"):
            print(f"    {tok.text} --{tok.dep_}--> {tok.head.text}")
    except OSError:
        print("spaCy 中文模型缺失，请运行:")
        print("  pip install https://github.com/explosion/spacy-models/releases/download/zh_core_web_sm-3.8.0/zh_core_web_sm-3.8.0-py3-none-any.whl")

In [ ]:
# 统一入口：定位项目根目录（notebook/ 的上一级），把 src/ 加入 import 路径
# 脚本产物路径保持原样（config.py 中的 BASE_DIR/outputs, BASE_DIR/data/...）
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 脚本产物写入的根目录（与 src/config.py 保持一致）
ARTIFACT_DIR = PROJECT_ROOT

# 默认使用 0 号 GPU；多卡机器可改成 "1" 等
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

print("项目根目录:", PROJECT_ROOT)
print("产物根目录:", ARTIFACT_DIR)

def run(cmd):
    """用当前 Python 解释器执行命令，工作目录为 PROJECT_ROOT。"""
    print(">>", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, env=os.environ.copy())

In [ ]:
import pandas as pd
from pathlib import Path

paths = [
    "data/raw/TRAIN/Train_reviews.csv",
    "data/raw/TRAIN/Train_labels.csv",
    "data/raw/TEST/Test_reviews.csv",
]
for p in paths:
    print(f"{p:45s} 存在: {Path(p).exists()}")

train = pd.read_csv("data/raw/TRAIN/Train_reviews.csv", encoding="utf-8")
labels = pd.read_csv("data/raw/TRAIN/Train_labels.csv", encoding="utf-8")
test = pd.read_csv("data/raw/TEST/Test_reviews.csv", encoding="utf-8")
print("训练评论:", train.shape, "| 训练标签:", labels.shape, "| 测试评论:", test.shape)
assert len(train) == 3229 and len(labels) == 6633 and len(test) == 2237, "数据规模与预期不一致，请检查原始数据"
display(train.head(3))

## Step 1. 查看规则配置

`config/model_baseline.yaml` 集中管理超参、情感种子、功能动词黑名单、否定词、转折/并列连词、类别关键词、依存关系集合等。脚本启动时自动加载；文件缺失时回退代码默认值。

In [ ]:
import yaml
cfg = yaml.safe_load(Path("config/model_baseline.yaml").read_text(encoding="utf-8"))
print("超参:", cfg["hyperparams"])
print("句法模型:", cfg["parser"])
lex = cfg["lexicons"]
print("手工情感种子数:", sum(len(v) for v in lex["manual_seed"].values()))
print("类别关键词覆盖类别数:", len(lex["manual_category_kw"]))

## Step 2. 运行完整基线

一条命令完成：**dev 评估 → 全量训练集归纳知识 → 测试集推理 → 写提交文件**。

```bash
python src/model_baseline.py
# 只想快速看 dev 指标、不生成提交文件：
python src/model_baseline.py --skip-test
```

In [ ]:
# 完整流程（约 1 分钟）
run([sys.executable, "src/model_baseline.py"])

# 仅 dev 评估时可改为：
# run([sys.executable, "src/model_baseline.py", "--skip-test"])

## Step 3. 查看 dev 指标与运行日志

In [ ]:
text = (ARTIFACT_DIR / "outputs/logs/baseline_report.txt").read_text(encoding="utf-8")
print(text)

## Step 4. 校验提交文件与归纳词典

`data/baseline/Result.csv` 为无表头 UTF-8 CSV，列顺序 `id,AspectTerms,OpinionTerms,Categories,Polarities`，空术语写 `_`。

In [ ]:
import pandas as pd
from pathlib import Path

COLS = ["id", "AspectTerms", "OpinionTerms", "Categories", "Polarities"]
CATS = {"整体", "使用体验", "功效", "价格", "物流", "气味", "包装",
        "真伪", "服务", "其他", "成分", "尺寸", "新鲜度"}
POLS = {"正面", "负面", "中性"}

def validate_result(path):
    p = Path(path)
    assert p.exists(), f"结果文件不存在: {p}"
    df = pd.read_csv(p, header=None, names=COLS, dtype=str)
    df["id"] = df["id"].str.strip().astype(int)
    test_ids = set(pd.read_csv("data/raw/TEST/Test_reviews.csv")["id"].astype(int))
    assert set(df["id"]) == test_ids, "测试集 id 覆盖不完整"
    assert set(df["Categories"].str.strip()) <= CATS, "出现非法类别"
    assert set(df["Polarities"].str.strip()) <= POLS, "出现非法极性"
    for c in COLS[1:]:
        df[c] = df[c].str.strip()
    print(f"[通过] {p}")
    print(f"  行数（四元组数）: {len(df)} | 覆盖 id: {df['id'].nunique()} | 密度: {len(df)/len(test_ids):.2f} 个/条")
    print("  类别分布:", dict(df["Categories"].value_counts()))
    print("  极性分布:", dict(df["Polarities"].value_counts()))
    return df

df = validate_result(ARTIFACT_DIR / "data/baseline/Result.csv")
display(df.head(10))

In [ ]:
asp_lex = pd.read_csv(ARTIFACT_DIR / "data/baseline/lexicon_aspect.csv")
opi_lex = pd.read_csv(ARTIFACT_DIR / "data/baseline/lexicon_opinion.csv")
print("方面词词典:", asp_lex.shape, "| 观点词词典:", opi_lex.shape)
display(opi_lex.head(10))
display(asp_lex.head(10))

## 结果解读

- 规则知识库的“分类”能力很强：**一旦 (A, O) 对抽对，类别准确率 96.79%、极性准确率 97.28%**；
- 瓶颈集中在“找对 / 配对”与隐式方面：dev 四元组 F1 只有 0.5535，明显低于后续学习型方案；
- 但规则基线完全可解释、无需 GPU，为整个项目提供了可靠的下限参照。